<a href="https://colab.research.google.com/github/shoh0806/Capstone_Design/blob/main/SASRec_Analysis_02_Positional_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#1. import

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch import optim
import math
import random
import time
import os

#2 SEED 고정

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Device: {device}")

Device: cuda


#3. 하이퍼파라미터

In [ ]:
DATA_DIR = "/content/drive/MyDrive"
max_len      = 100
hidden_dim   = 100
num_heads    = 4
num_layers   = 4
batch_size   = 128
LR           = 0.001
weight_decay = 0.0001
epochs       = 30
stride       = 10
patience     = 10

#4. 데이터 전처리 및 데이터 분리

In [ ]:
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
ratings = ratings.sort_values(by=["userId", "timestamp"])

user_seq = ratings.groupby("userId")["movieId"].apply(list)

item_set  = sorted(ratings["movieId"].unique())
item2idx  = {item: i+1 for i, item in enumerate(item_set)}  # 1-based, 0은 padding
idx2item  = {i: item for item, i in item2idx.items()}

lengths = user_seq.apply(len)
print(f"평균: {lengths.mean():.1f}  최대: {lengths.max()}  최소: {lengths.min()}  중앙값: {lengths.median()}")

# 유저별 시퀀스를 idx로 변환
user_sequences = [[item2idx[i] for i in seq] for seq in user_seq]


# Train / Val / Test 분리 (leave-two-out)
train_sequences = []
val_data        = []
test_data       = []

for seq in user_sequences:
    if len(seq) < 20:
        continue
    train = seq[:-2]
    val   = seq[-2]
    test  = seq[-1]

    train_sequences.append(train)
    val_data.append((train, [val]))
    test_data.append((train + [val], [test]))

print(f"학습 유저 수: {len(train_sequences):,}")

평균: 165.6  최대: 2314  최소: 20  중앙값: 96.0
학습 유저 수: 6,040


#5. Dataset


In [ ]:
class SASRecDataset(Dataset):
    def __init__(self, sequences, max_len, stride):
        self.data     = []
        self.max_len  = max_len
        # 전체 아이템 집합 (negative sampling용)
        all_items = set()
        for seq in sequences:
            all_items.update(seq)
        self.all_items = list(all_items)  # 루프 밖에서 한 번만


        for seq in sequences:
            for i in range(stride, len(seq), stride):
                window = seq[max(0, i - max_len): i + 1]

                input_seq = window[:-1]
                target    = window[1:]

                pad_len   = max_len - len(input_seq)
                input_seq = [0] * pad_len + input_seq
                target    = [0] * pad_len + target

                # 위치별 negative 1개씩 샘플링
                seq_set = set(seq)
                neg = []
                for t in target:
                    if t == 0:
                        neg.append(0)
                    else:
                        n = random.choice(self.all_items)
                        while n in seq_set:          # 이미 본 아이템 제외
                            n = random.choice(self.all_items)
                        neg.append(n)                # while 밖에서 append

                self.data.append((input_seq, target, neg))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_seq, target, neg = self.data[idx]
        return (torch.LongTensor(input_seq),
                torch.LongTensor(target),
                torch.LongTensor(neg))


dataset = SASRecDataset(train_sequences, max_len, stride)
loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True)
print(f"학습 샘플 수: {len(dataset):,}")

학습 샘플 수: 95,459


#6. Model

In [ ]:
class SASRecBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=0.2,
            batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )
        self.norm1   = nn.LayerNorm(hidden_dim)
        self.norm2   = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x, attn_mask, padding_mask):
        attn_output, attn_weights = self.attn(
            x, x, x,
            attn_mask=attn_mask,
            key_padding_mask=padding_mask,
            need_weights=True,
            average_attn_weights=False
            # no positional = True?
        )
        attn_output = self.dropout(attn_output)
        x = self.norm1(x + attn_output)

        ffn_output = self.ffn(x)
        ffn_output = self.dropout(ffn_output)
        x = self.norm2(x + ffn_output)

        return x, attn_weights


class SASRec(nn.Module):
    def __init__(self, num_items, hidden_dim, max_len, num_heads, num_layers):
        super().__init__()
        self.item_emb = nn.Embedding(num_items + 1, hidden_dim, padding_idx=0)
        self.pos_emb  = nn.Embedding(max_len, hidden_dim)
        self.layers   = nn.ModuleList([
            SASRecBlock(hidden_dim, num_heads) for _ in range(num_layers)
        ])
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout    = nn.Dropout(0.2)

    def forward(self, batch_input):
        batch_size, seq_len = batch_input.size()

        pos = torch.arange(seq_len, device=batch_input.device).unsqueeze(0).repeat(batch_size, 1)

        x = self.item_emb(batch_input) * math.sqrt(self.item_emb.embedding_dim)
        x = x + self.pos_emb(pos)
        x = self.dropout(x)

        # Causal mask
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1)
        mask = mask.masked_fill(mask == 1, -1e9).float()

        # Padding mask
        padding_mask = (batch_input == 0)
        x = x.masked_fill(padding_mask.unsqueeze(-1), 0)

        attn_weights_all = []
        for layer in self.layers:
            x, attn_weights = layer(x, mask, padding_mask)  #SASRecBlock.forward도 자동 실행됨
            attn_weights_all.append(attn_weights)

        x = self.layer_norm(x)
        return x, attn_weights_all


model = SASRec(len(item2idx), hidden_dim, max_len, num_heads, num_layers).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/sasrec_best.pt"))
model.eval()
print("모델 로드 완료!")


모델 로드 완료!


# 7 평가함수

In [ ]:
def evaluate(model, data, top_k=10, batch_size=256):
    model.eval()
    HR, NDCG = 0, 0

    with torch.no_grad():
        for i in range(0, len(data), batch_size):
            batch = data[i : i + batch_size]   # 유저 여러 명 묶기

            seqs, gts = [], []
            for seq, gt in batch:
                s = seq[-max_len:]
                s = [0] * (max_len - len(s)) + s
                seqs.append(s)
                gts.append(gt[0])

            inp = torch.LongTensor(seqs).to(device)  # shape: (B, 100)
            output, _ = model(inp)                    # forward 1번으로 B명 처리
            h_last = output[:, -1, :]                 # (B, D)

            all_emb = model.item_emb.weight           # (num_items+1, D)
            scores = torch.matmul(h_last, all_emb.T)  # (B, num_items+1)

            scores[:, 0] = -1e9   # padding 제외

            for j, (seq, gt_idx) in enumerate(zip([d[0] for d in batch], gts)):
                for seen in seq:
                    scores[j, seen] = -1e9

                rank = (scores[j] > scores[j, gt_idx]).sum().item()
                if rank < top_k:
                    HR   += 1
                    NDCG += 1 / math.log2(rank + 2)

    return HR / len(data), NDCG / len(data)

# 8. 실험 02 - Positional Encoding 유무 비교

In [ ]:
def evaluate_no_pos(model, data, top_k=10, batch_size=256):
    model.eval()
    HR, NDCG = 0, 0

    with torch.no_grad():
        for i in range(0, len(data), batch_size):
            batch = data[i : i + batch_size]

            seqs, gts = [], []
            for seq, gt in batch:
                s = seq[-max_len:]
                s = [0] * (max_len - len(s)) + s
                seqs.append(s)
                gts.append(gt[0])

            inp = torch.LongTensor(seqs).to(device)
            batch_size_actual, seq_len = inp.size()

            # 포지셔널 인코딩 없이 forward
            x = model.item_emb(inp) * math.sqrt(model.item_emb.embedding_dim)
            # x = x + model.pos_emb(pos)  ← 이 줄을 생략
            x = model.dropout(x)

            mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1)
            mask = mask.masked_fill(mask == 1, -1e9).float()
            padding_mask = (inp == 0)
            x = x.masked_fill(padding_mask.unsqueeze(-1), 0)

            for layer in model.layers:
                x, _ = layer(x, mask, padding_mask)
            x = model.layer_norm(x)

            h_last = x[:, -1, :]
            all_emb = model.item_emb.weight
            scores = torch.matmul(h_last, all_emb.T)
            scores[:, 0] = -1e9

            for j, (seq, gt_idx) in enumerate(zip([d[0] for d in batch], gts)):
                for seen in seq:
                    scores[j, seen] = -1e9
                rank = (scores[j] > scores[j, gt_idx]).sum().item()
                if rank < top_k:
                    HR   += 1
                    NDCG += 1 / math.log2(rank + 2)

    return HR / len(data), NDCG / len(data)

# 원본 vs 포지셔널 인코딩 제거 비교
hr_orig, ndcg_orig = evaluate(model, test_data)
hr_npos, ndcg_npos = evaluate_no_pos(model, test_data)

print(f"[원본 (PE 있음)] HR@10: {hr_orig:.4f}  NDCG@10: {ndcg_orig:.4f}")
print(f"[PE 제거]        HR@10: {hr_npos:.4f}  NDCG@10: {ndcg_npos:.4f}")
print(f"\nHR 변화량:   {hr_orig - hr_npos:.4f}")
print(f"NDCG 변화량: {ndcg_orig - ndcg_npos:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


[원본 (PE 있음)] HR@10: 0.2376  NDCG@10: 0.1272
[PE 제거]        HR@10: 0.2073  NDCG@10: 0.1083

HR 변화량:   0.0303
NDCG 변화량: 0.0189
